# Experiment 13 — Install Hadoop and Explore HDFS

**Aim:** Install Hadoop in a WSL Ubuntu user-local environment and demonstrate basic HDFS file operations.

**Libraries / software:** Apache Hadoop 3.4.2, Eclipse Temurin Java 17, WSL 2 with Ubuntu, and a small local text file. Hadoop and Java are installed under the WSL user's home; no administrator packages or credentials are used.

**Theory:** HDFS provides a filesystem namespace backed by NameNode metadata and DataNode blocks. This single-machine pseudo-distributed setup is for learning only and is not a production cluster.

**Procedure / Implementation:** The executed cell configures a local NameNode/DataNode, uploads a harmless sample file, lists and reads it, downloads a copy, verifies the contents, and stops the daemons.

**Environment note:** The Hadoop release archive and Java runtime are installed by the setup commands documented below. The lab notebook invokes the user's WSL installation through `wsl.exe`.

## Procedure / Implementation — Install Hadoop in WSL Ubuntu
Run these commands in WSL to install the same user-local versions used for this execution. Verify the Hadoop SHA-512 before extracting; the commands do not require `sudo`.

```bash
mkdir -p "$HOME/.local/opt" "$HOME/.cache/dsa-lab"
curl -fL "https://api.adoptium.net/v3/binary/latest/17/ga/linux/x64/jdk/hotspot/normal/eclipse" \
  -o "$HOME/.cache/dsa-lab/temurin17.tar.gz"
mkdir -p "$HOME/.local/opt/jdk-17"
tar -xzf "$HOME/.cache/dsa-lab/temurin17.tar.gz" \
  -C "$HOME/.local/opt/jdk-17" --strip-components=1
curl -fL "https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz" \
  -o "$HOME/.cache/dsa-lab/hadoop-3.4.2.tar.gz"
curl -fsSL "https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz.sha512" \
  -o "$HOME/.cache/dsa-lab/hadoop-3.4.2.tar.gz.sha512"
cd "$HOME/.cache/dsa-lab"
sha512sum -c hadoop-3.4.2.tar.gz.sha512
mkdir -p "$HOME/.local/opt/hadoop-3.4.2"
tar -xzf hadoop-3.4.2.tar.gz -C "$HOME/.local/opt/hadoop-3.4.2" --strip-components=1
```

The archive is large. Use official download sources, confirm storage/network availability, and record the versions actually installed.

## 1. Record platform and Hadoop versions

In [1]:
import subprocess

def run_wsl(command, check=True, echo=True):
    completed = subprocess.run(
        ["wsl.exe", "--exec", "bash", "-lc", command],
        capture_output=True,
        text=True,
    )
    if echo and completed.stdout:
        print(completed.stdout, end="")
    if echo and completed.stderr:
        print(completed.stderr, end="")
    if check and completed.returncode:
        raise RuntimeError(
            f"WSL command failed with exit code {completed.returncode}: "
            f"{completed.stderr or completed.stdout}"
        )
    return completed

HADOOP_SETUP = r'''
export JAVA_HOME="$HOME/.local/opt/jdk-17"
export HADOOP_HOME="$HOME/.local/opt/hadoop-3.4.2"
export PATH="$JAVA_HOME/bin:$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"
'''

print("WSL distribution:", run_wsl("cat /etc/os-release | head -n 1").stdout.strip())
run_wsl(HADOOP_SETUP + "java -version 2>&1; hadoop version | head -n 5")

PRETTY_NAME="Ubuntu 26.04 LTS"
WSL distribution: PRETTY_NAME="Ubuntu 26.04 LTS"


openjdk version "17.0.21" 2026-10-20
OpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)
OpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)
Hadoop 3.4.2
Source code repository https://github.com/apache/hadoop.git -r 84e8b89ee2ebe6923691205b9e171badde7a495c
Compiled by ahmarsu on 2025-08-20T10:30Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4


CompletedProcess(args=['wsl.exe', '--exec', 'bash', '-lc', '\nexport JAVA_HOME="$HOME/.local/opt/jdk-17"\nexport HADOOP_HOME="$HOME/.local/opt/hadoop-3.4.2"\nexport PATH="$JAVA_HOME/bin:$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"\njava -version 2>&1; hadoop version | head -n 5'], returncode=0, stdout='openjdk version "17.0.21" 2026-10-20\nOpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)\nOpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)\nHadoop 3.4.2\nSource code repository https://github.com/apache/hadoop.git -r 84e8b89ee2ebe6923691205b9e171badde7a495c\nCompiled by ahmarsu on 2025-08-20T10:30Z\nCompiled on platform linux-x86_64\nCompiled with protoc 3.23.4\n', stderr='')

## 2. Configure and exercise HDFS

In [2]:
from pathlib import Path
import shlex

ROOT = Path.cwd()
SAMPLE = ROOT / "datasets" / "hdfs_practical_sample.txt"
SAMPLE.parent.mkdir(parents=True, exist_ok=True)
sample_contents = (
    "IT-411 Hadoop practical sample\n"
    "HDFS stores files in a distributed namespace.\n"
    "This file contains no personal or confidential data.\n"
)
SAMPLE.write_text(sample_contents, encoding="utf-8")

home = run_wsl("printf %s \"$HOME\"").stdout.strip()
hadoop_home = f"{home}/.local/opt/hadoop-3.4.2"
name_dir = f"{home}/.local/var/hdfs/name"
data_dir = f"{home}/.local/var/hdfs/data"
wsl_sample = f"{home}/.local/var/hdfs/{SAMPLE.name}"
wsl_copy = f"{home}/.local/var/hdfs/exp13_downloaded.txt"
sample_b64 = __import__("base64").b64encode(
    sample_contents.encode("utf-8")
).decode("ascii")
run_wsl(
    f"mkdir -p {shlex.quote(home + '/.local/var/hdfs')}; "
    + f"printf %s {shlex.quote(sample_b64)} | base64 -d > {shlex.quote(wsl_sample)}"
)

core_site = f'''<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
</configuration>
'''
hdfs_site = f'''<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.namenode.name.dir</name><value>file://{name_dir}</value></property>
  <property><name>dfs.datanode.data.dir</name><value>file://{data_dir}</value></property>
</configuration>
'''

for filename, content in [
    ("core-site.xml", core_site),
    ("hdfs-site.xml", hdfs_site),
]:
    target = f"{hadoop_home}/etc/hadoop/{filename}"
    write = f"cat > {shlex.quote(target)} <<'IT411_EOF'\n{content}IT411_EOF"
    run_wsl(HADOOP_SETUP + write)

env_setup = (
    HADOOP_SETUP
    + f"export HADOOP_CONF_DIR={shlex.quote(hadoop_home + '/etc/hadoop')}; "
)
format_cmd = (
    env_setup
    + f"mkdir -p {shlex.quote(name_dir)} {shlex.quote(data_dir)}; "
    + f"if [ ! -d {shlex.quote(name_dir + '/current')} ]; then "
    + "hdfs namenode -format -force; fi"
)
run_wsl(format_cmd)

remote_user = run_wsl("printf %s \"$USER\"").stdout.strip()
destination = f"/user/{remote_user}/exp13/"
exercise = (
    env_setup
    + r'''
set -e
LOG_DIR="$HADOOP_HOME/logs"
mkdir -p "$LOG_DIR"
cleanup() {
  hdfs --daemon stop datanode >/dev/null 2>&1 || true
  hdfs --daemon stop namenode >/dev/null 2>&1 || true
}
trap cleanup EXIT
hdfs --daemon start namenode
hdfs --daemon start datanode
ready=false
for attempt in $(seq 1 60); do
  report="$(hdfs dfsadmin -report 2>&1)" || report=""
  if printf '%s' "$report" | grep -q 'Live datanodes (1)'; then
    printf '%s\n' "$report"
    ready=true
    break
  fi
  sleep 1
done
if [ "$ready" != true ]; then
  printf '%s\n' "$report"
  echo "HDFS did not become ready within 60 seconds." >&2
  exit 1
fi
'''
    + "hdfs dfs -mkdir -p "
    + shlex.quote(destination)
    + "; hdfs dfs -rm -f "
    + shlex.quote(destination + "exp13_sample.txt")
    + " >/dev/null 2>&1 || true; hdfs dfs -put -f "
    + shlex.quote(wsl_sample)
    + " "
    + shlex.quote(destination)
    + "; echo 'HDFS directory listing:'; hdfs dfs -ls "
    + shlex.quote(destination)
    + "; echo 'HDFS file contents:'; hdfs dfs -cat "
    + shlex.quote(destination + SAMPLE.name)
    + "; hdfs dfs -get -f "
    + shlex.quote(destination + SAMPLE.name)
    + " "
    + shlex.quote(wsl_copy)
    + "; echo 'Remove the HDFS test file:'; hdfs dfs -rm -f "
    + shlex.quote(destination + SAMPLE.name)
)
run_wsl(exercise)
downloaded = run_wsl("cat " + shlex.quote(wsl_copy)).stdout
assert downloaded == sample_contents
print("Downloaded file matches the local source:", downloaded == sample_contents)
print("HDFS daemons stopped by the cleanup trap.")

/home/kritik

kritik

Configured Capacity: 1081101176832 (1006.85 GB)
Present Capacity: 1019754168461 (949.72 GB)
DFS Remaining: 1019754143744 (949.72 GB)
DFS Used: 24717 (24.14 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (1):

Name: 127.0.0.1:9866 (localhost)
Hostname: Kritik.localdomain
Decommission Status : Normal
Configured Capacity: 1081101176832 (1006.85 GB)
DFS Used: 24717 (24.14 KB)
Non DFS Used: 6354653043 (5.92 GB)
DFS Remaining: 1019754143744 (949.72 GB)
DFS Used%: 0.00%
DFS Remaining%: 94.33%
Configured Cache 

IT-411 Hadoop practical sample
HDFS stores files in a distributed namespace.
This file contains no personal or confidential data.
Downloaded file matches the local source: True
HDFS daemons stopped by the cleanup trap.


## Observation
The executed cells report the WSL/Java/Hadoop versions, HDFS listing and file contents, and a local-versus-HDFS round-trip check.

## Result
HDFS operations are considered successful only if the upload/read/download verification above completes. This is a one-machine learning setup, not evidence of a multi-node production deployment.

## User-local installation commands
The environment uses a Java 17 JDK and Hadoop 3.4.2 extracted under `~/.local/opt/`. The release archive and checksum should be verified before unpacking. The notebook configures only its HDFS data directory under `~/.local/var/hdfs/`.